In [1]:
import numpy as np
from scipy import optimize
import scipy as sp

import matplotlib.pyplot as plt
import matplotlib.axes as axe
import pandas as pd
import datetime as dt
import gurobipy as gp
from gurobipy import GRB
import cvxpy as cp
import yaml

import random
from itertools import chain, combinations, product
import time

import glob
import os

plt.rcParams['text.usetex'] = True

# Functions

In [2]:
# def poly(alpha, x, p=4):
#     assert alpha.shape[0] == p+1,  "alpha should be a length-(p+1) vector."
#     assert np.all(alpha > 0.0), "All entries of alpha should be strictly positive."
#     assert x >= 0.0, "x should be non-negative."
    
#     return sum(alpha[k] * (x**k) for k in range(p))

# def poly_simplified(alpha, x, p=4):
#     assert alpha.shape[0] == p+1,  "alpha should be a length-(p+1) vector."
#     assert np.all(alpha > 0.0), "All entries of alpha should be strictly positive."
#     assert x >= 0.0, "x should be non-negative."
    
#     return alpha[0] + alpha[p-1] * (x**(p-1))

# def route_latency(alphas, xs, edges, p=4):
#     assert len(alphas.shape) == 2, "alphas should be a 2D array."
#     assert alphas.shape[0] == xs.shape[0], "The first dimension of alphas should equal the dimension of xs."
#     assert set(edges).issubset(range(alphas.shape[0])), "edges should be a subset of indices of edges"
    
#     return sum([poly(alphas[e, :], xs[e], p=4) for e in edges])


In [57]:
def reshape_vec(v):

    # print("length of v to reshape:", v.shape[0])

    len_v = v.shape[0]
    v_reshaped = v.reshape((len_v, 1))
    
    return v_reshaped


In [114]:
# DR: distributionally robust

def generate_R_from_edge_node_matrix(edge_node_matrix):
    assert len(edge_node_matrix.shape) == 2
    assert edge_node_matrix.shape[1] == 2
    
    num_edges = edge_node_matrix.shape[0]
    num_nodes = np.max(edge_node_matrix) + 1
    
    R = np.zeros((num_nodes - 1, num_edges))
    
    for node_index in range(R.shape[0]):
        for edge_index in range(R.shape[1]):
            if edge_node_matrix[edge_index][0] == node_index:
                R[node_index, edge_index] = 1
            elif edge_node_matrix[edge_index][1] == node_index:
                R[node_index, edge_index] = -1

    return R


def generate_Gamma(B, R):

    inv_B = np.linalg.inv(B)
    R_inv_B_R_top = np.linalg.inv(R @ inv_B @ R.T)
    
    return inv_B - inv_B @ R.T @ R_inv_B_R_top @ R @ inv_B


def compute_eps_max(Gamma, B, R, alpha_mean_nominal, eta, delta):
    
    # print()
    # print("Gamma.shape", Gamma.shape)
    # print("alpha_mean_nominal.shape:", alpha_mean_nominal.shape)
    # print()
    
    num_nodes = R.shape[0] + 1
    num_edges = R.shape[1]
    Gamma_norm = np.linalg.norm(Gamma, ord=2)
    inv_B = np.linalg.inv(B)
    R_inv_B_R_top = np.linalg.inv(R @ inv_B @ R.T)
    eta_vec = eta * np.concatenate(([1], np.zeros(num_nodes-2)))
    offset_with_eta = inv_B @ R.T @ R_inv_B_R_top @ eta_vec

    # Define and solve the CVXPY problem.
    eps = cp.Variable(1)
    tau = cp.Variable(num_edges)
    func = eps
    obj = cp.Maximize(func)
    constr = [Gamma @ tau <= - Gamma_norm * (eps + delta) * np.ones(num_edges) \
                 - Gamma @ alpha_mean_nominal + offset_with_eta]
    constr += [tau >= 0.0]
    constr += [eps >= 0.0]
    prob = cp.Problem(obj, constr)
    prob.solve()

    # # Print result.
    # print()
    # print("eps_max:", eps.value[0])
    # print()

    return eps.value
    

def compute_opt_DR_toll_constrained(Gamma, B, R, alpha_mean_nominal, eta, eps, delta):
    
    num_nodes = R.shape[0] + 1
    num_edges = R.shape[1]
    Gamma_norm = np.linalg.norm(Gamma, ord=2)
    inv_B = np.linalg.inv(B)
    R_inv_B_R_top = np.linalg.inv(R @ inv_B @ R.T)
    eta_vec = eta * np.concatenate(([1], np.zeros(num_nodes-2)))
    offset_with_eta = inv_B @ R.T @ R_inv_B_R_top @ eta_vec
    
    tau = cp.Variable(num_edges)
    func = eps * cp.norm(Gamma @ tau + offset_with_eta) \
            + cp.quad_form(tau, Gamma) + alpha_mean_nominal.T @ Gamma @ tau
#     func = cp.quad_form(tau, Gamma) + alpha_mean_nominal.T @ Gamma @ tau
    obj = cp.Minimize(func)
    constr = [Gamma @ tau <= - Gamma_norm * (eps + delta) * np.ones(num_edges) \
                 - Gamma @ alpha_mean_nominal + offset_with_eta]
    constr += [tau >= 0.0]
    
    prob = cp.Problem(obj, constr)
    prob.solve()

    # # Print result.
    # print()
    # print("DR optimal toll:", tau.value)
    # print()
    # print("Worst-case DR latency:", prob.value)
    # print()
    
    return tau.value, prob.value

def compute_opt_DR_toll(Gamma, B, R, alpha_mean_nominal, eta, eps, delta):
    
    num_nodes = R.shape[0] + 1
    num_edges = R.shape[1]
    Gamma_norm = np.linalg.norm(Gamma, ord=2)
    inv_B = np.linalg.inv(B)
    R_inv_B_R_top = np.linalg.inv(R @ inv_B @ R.T)
    eta_vec = eta * np.concatenate(([1], np.zeros(num_nodes-2)))
    offset_with_eta = inv_B @ R.T @ R_inv_B_R_top @ eta_vec
    
    tau = cp.Variable(num_edges)
    func = eps * cp.norm(Gamma @ tau + offset_with_eta) \
            + cp.quad_form(tau, Gamma) + alpha_mean_nominal.T @ Gamma @ tau
#     func = cp.quad_form(tau, Gamma) + alpha_mean_nominal.T @ Gamma @ tau
    obj = cp.Minimize(func)
    # constr = [Gamma @ tau <= - Gamma_norm * (eps + delta) * np.ones(num_edges) \
    #              - Gamma @ alpha_mean_nominal + offset_with_eta]
    constr = []
    constr += [tau >= 0.0]
    
    prob = cp.Problem(obj, constr)
    prob.solve()

    # # Print result.
    # print()
    # print("DR optimal toll:", tau.value)
    # print()
    # print("Worst-case DR latency:", prob.value)
    # print()
    
    return tau.value, prob.value


def latency_eq_worst(B, R, alpha, tau, eta):
    num_nodes = R.shape[0] + 1
    eta_vec = eta * np.concatenate(([1], np.zeros(num_nodes-2)))
    
    tau = reshape_vec(tau)
    alpha = reshape_vec(alpha)
    eta_vec = reshape_vec(eta_vec)
    
    BRR_matrix = np.block([[B, R.T], [R, np.zeros((num_nodes-1, num_nodes-1))]])

    return np.block([tau.T, eta_vec.T]) @ np.linalg.inv(BRR_matrix) @ np.block([[alpha + tau], [- eta_vec]])

    
def compute_worst_theta(B, R, tau, eta, alpha_mean_nominal, eps):

    num_nodes = R.shape[0] + 1
    num_edges = R.shape[1]
    
    BRR_matrix = np.block([[B, R.T], [R, np.zeros((num_nodes-1, num_nodes-1))]])
    eta_vec = eta * np.concatenate(([1], np.zeros(num_nodes-2)))

    # print("tau.shape:", tau.shape)
    # print("eta_vec.shape", eta_vec.shape)
    
    tau_eta = np.append(tau, eta_vec)

    # print("np.block([np.identity(num_edges), np.zeros((num_nodes-1, num_edges))]).shape:", \
    #      np.block([np.identity(num_edges), np.zeros((num_nodes-1, num_edges))]).shape)
    # print("BRR_matrix.shape:", BRR_matrix.shape)
    # print("eta_vec.shape", eta_vec.shape)
    
    q_tau = np.block([np.identity(num_edges), np.zeros((num_edges, num_nodes-1))]) @ np.linalg.inv(BRR_matrix) @ tau_eta
    
    theta_worst = alpha_mean_nominal + eps * q_tau / np.linalg.norm(q_tau)
    
    return theta_worst

def sample_from_ball(center, radius, n_samples=1):
    center = np.asarray(center)
    d = center.shape[0]
    
    # Step 1: random directions
    x = np.random.normal(size=(n_samples, d))
    x /= np.linalg.norm(x, axis=1, keepdims=True)
    
    # Step 2: radii
    u = np.random.uniform(0, 1, size=(n_samples, 1))
    r = radius * (u ** (1.0 / d))
    
    # Step 3: combine
    samples = center + r * x
    
    return samples

def latency_eq_worst_under_distribution_shift(B, R, alpha_mean_nominal, tau, eta, eps):

    num_nodes = R.shape[0] + 1
    eta_vec = eta * np.concatenate(([1], np.zeros(num_nodes-2)))
    
    tau = reshape_vec(tau)
    alpha_mean_nominal = reshape_vec(alpha_mean_nominal)
    eta_vec = reshape_vec(eta_vec)

    Gamma = generate_Gamma(B, R)
    B_inv = np.linalg.inv(B)
    R_B_inv_R_T = R @ B_inv @ R.T

    vec_temp = Gamma @ tau + B_inv @ R.T @ np.linalg.inv(R_B_inv_R_T) @ eta_vec

    return eps * np.linalg.norm(vec_temp) + tau.T @ Gamma @ tau + alpha_mean_nominal.T @ Gamma @ tau
    


In [82]:
# # a = np.array([1, 2, 3]).reshape((3, 1))
# # b = np.array([4, 5]).reshape((2, 1))

# a = np.array([1, 2, 3])
# b = np.array([4, 5])

# # np.block([[a], [b]])
# np.block([a, b])

# np.append([1, 2, 3], [2, 4])
# np.identity(3)
# np.block([np.identity(2), np.identity(2)])

# Sioux Falls Network

In [6]:
num_nodes = 7
num_edges = 10

In [7]:
offsets_orig = np.array([1.33, 1.20, 2.77, 1.48, 1.00, 0.70, 0.89, 1.34, 1.91, 1.61])
quartic_coefficients_orig = np.array([0.02, 0.02, 0.03, 0.03, 0.01, 0.02, 0.01, 0.04, 0.03, 0.04])
num_travelers_orig = 15

# Version 1:

num_travelers_to_use_for_offset = 3
num_travelers_to_use = 9
beta = np.zeros(num_edges)
alpha_mean_nominal = [offsets_orig[e] + quartic_coefficients_orig[e] * num_travelers_to_use_for_offset**4 for e in range(num_edges)]

for e in range(num_edges):
    traveler_count_LS = np.array(list(range(num_travelers_to_use+1)))
    latency_data_LS = np.array([offsets_orig[e] + quartic_coefficients_orig[e] * (k**4) for k in traveler_count_LS])
    beta[e] = np.dot(traveler_count_LS, latency_data_LS - alpha_mean_nominal) / np.dot(traveler_count_LS, traveler_count_LS)
    # print(beta[e])

alpha_mean_nominal = np.array(alpha_mean_nominal)
    
print("beta:\n", beta)
print()
print("alpha_mean_nominal:\n", alpha_mean_nominal)

## Version 2: Affine Regression:

# num_travelers_to_use = 5
# beta = np.zeros(num_edges)
# alpha_mean_nominal = np.zeros(num_edges)

# for e in range(num_edges):
#     traveler_count_LS = np.array(list(range(num_travelers_to_use+1)))
#     latency_data_LS = np.array([offsets_orig[e] + quartic_coefficients_orig[e] * (k**4) for k in traveler_count_LS])
#     data_matrix_LS = np.vstack([traveler_count_LS, np.ones(len(traveler_count_LS))]).T
#     beta[e], alpha_mean_nominal[e] = np.linalg.lstsq(data_matrix_LS, latency_data_LS, rcond=None)[0]

# print(beta)
# print()
# print(alpha_mean_nominal)

## Version 3: Straight line connecting points:

# assert alpha.shape[0] == num_edges
# assert quartic_coefficients.shape[0] == num_edges
# # Maximum num_players_affine_approx = 15
# num_players_affine_approx = 10
# beta = np.array([quartic_coefficients_orig[i] * num_players_affine_approx**(4-1) for i in range(num_edges)])
# alpha_mean_nominal[e] = offsets_orig[e]
# print(beta)


beta:
 [ 8.11203509  8.09150877 12.57887719 12.37519298  3.82045614  8.0125614
  3.80308772 16.5925614  12.44308772 16.63519298]

alpha_mean_nominal:
 [2.95 2.82 5.2  3.91 1.81 2.32 1.7  4.58 4.34 4.85]


In [8]:
# e = 0

# traveler_count_line = np.linspace(np.min(traveler_count_LS), np.max(traveler_count_LS), 100)
# latency_line = beta[e] * traveler_count_line + alpha_mean_nominal[e]

# plt.scatter(traveler_count_LS, latency_data_LS, color='blue', label='Data points')
# plt.plot(traveler_count_line, latency_line, color='red', label=f'Best fit: y={beta[e]:.2f}x+{alpha_mean_nominal[e]:.2f}')

# plt.xlabel("Traveler Count")
# plt.ylabel("Latency")
# plt.title("Least Squares Best-Fit Line")
# plt.legend()
# plt.grid(True)

# plt.show()



In [44]:
alpha_mean_nominal

array([2.95, 2.82, 5.2 , 3.91, 1.81, 2.32, 1.7 , 4.58, 4.34, 4.85])

In [88]:
B = np.diag(beta)
edge_node_matrix = np.array([[0, 2], [0, 1], [2, 6], [2, 3], [1, 3], \
                             [1, 4], [3, 5], [4, 6], [4, 5], [5, 6]])

delta = 0.2
eta = 2.0

R = generate_R_from_edge_node_matrix(edge_node_matrix)
Gamma = generate_Gamma(B, R)
eps_max_original = compute_eps_max(Gamma, B, R, alpha_mean_nominal, eta, delta)[0]
eps_max_adjusted = max(min(eps_max_original, min(alpha_mean_nominal) - delta), 0.0)

print("eps_max_adjusted:", eps_max_adjusted)
print("eps_max_original:", eps_max_original)

# Compute distributionally robust toll and corresponding worst-case latency (in theory):

eps = 0.5 * eps_max_adjusted
tau_opt, worst_case_ = compute_opt_DR_toll_constrained(Gamma, B, R, alpha_mean_nominal, eta, eps, delta)

# Compute certainty equivalence toll and corresponding worst-case latency (in theory):



eps_max_adjusted: 1.5000000000000002
eps_max_original: 2.3308301972996186


## Main Code:

In [108]:
num_eps = 5
# eps_set = np.linspace(0.0, eps_max_adjusted, num = num_eps)
eps_set = np.linspace(0.0, eps_max_original, num = num_eps)
num_samples = 10000

worst_latency_array = np.zeros((num_eps, num_eps))
tau_array = np.zeros((num_eps, num_edges))
alpha_mean_array = np.zeros((num_eps, num_eps, num_edges))

# for eps_true, eps_hat in product(eps_set, repeat=2):
#     print(eps_true, eps_hat)

for i, eps_true in enumerate(eps_set):
    for j, eps_hat in enumerate(eps_set):
        # tau_eps_hat, _ = compute_opt_DR_toll_constrained(Gamma = Gamma,\
        #                                      B = B,\
        #                                      R = R,\
        #                                      alpha_mean_nominal = alpha_mean_nominal,\
        #                                      eta = eta,\
        #                                      eps = eps_hat, \
        #                                      delta = delta)

        tau_eps_hat, _ = compute_opt_DR_toll(Gamma = Gamma,\
                                     B = B,\
                                     R = R,\
                                     alpha_mean_nominal = alpha_mean_nominal,\
                                     eta = eta,\
                                     eps = eps_hat, \
                                     delta = delta)
        tau_array[j, :] = tau_eps_hat
        
        # print("tau_eps_hat:", tau_eps_hat)
        alpha_mean_eps = compute_worst_theta(B = B,\
                                             R = R,\
                                             tau = tau_eps_hat,\
                                             eta = eta,\
                                             alpha_mean_nominal = alpha_mean_nominal,\
                                             eps = eps_true)
        alpha_mean_array[i, j, :] = alpha_mean_eps
        alpha_vals = sample_from_ball(alpha_mean_eps, delta, n_samples = num_samples).T
        

        latency_array_temp = np.zeros(num_samples)
        for n in range(num_samples):
            latency_array_temp[n] = latency_eq_worst(B,\
                                                     R,\
                                                     alpha_vals[:, n],\
                                                     tau_eps_hat,\
                                                     eta)
        worst_latency_array[i, j] = np.mean(latency_array_temp)
        
        # worst_latency_array[i, j] = s
        # break


# def compute_opt_DR_toll(Gamma, B, R, alpha_mean_nominal, eta, eps, delta):
# def compute_worst_theta(B, R, tau, eta, alpha_mean_nominal, eps):
# def latency_eq_worst(B, R, alpha, tau, eta):


/var/folders/wc/5q980jhx25l5t8_jtg5hp94m0000gn/T/ipykernel_1748/2893080940.py:45: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  latency_array_temp[n] = latency_eq_worst(B,\


In [109]:
# print("alpha_vals.shape:", alpha_vals.shape)
worst_latency_array
# eps_set

array([[59.39888127, 59.39719676, 59.39933935, 59.40034339, 59.40332611],
       [60.60336278, 60.6058833 , 60.60437206, 60.60723767, 60.60912209],
       [61.811062  , 61.80892292, 61.81105989, 61.81230559, 61.81243918],
       [63.01870717, 63.01678319, 63.01369565, 63.01472807, 63.0148318 ],
       [64.22606771, 64.220833  , 64.219223  , 64.21931301, 64.21892819]])

In [100]:
# tau_array

In [115]:
worst_latency_array_from_observations = np.zeros((num_eps, num_eps))

for i, eps_true in enumerate(eps_set):
    for j, eps_hat in enumerate(eps_set):
        worst_latency_array_from_observations[i, j] = \
            latency_eq_worst_under_distribution_shift(B = B,\
                                                      R = R,\
                                                      alpha_mean_nominal = alpha_mean_nominal,\
                                                      tau = tau_array[j],\
                                                      eta = eta,
                                                      eps = eps_true)

# def latency_eq_worst_under_distribution_shift(B, R, alpha_mean_nominal, tau, eta, eps):

/var/folders/wc/5q980jhx25l5t8_jtg5hp94m0000gn/T/ipykernel_1748/814734142.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  worst_latency_array_from_observations[i, j] = \


In [116]:
worst_latency_array_from_observations

array([[-0.27578416, -0.27537052, -0.27419737, -0.27236035, -0.26992788],
       [ 0.93104213,  0.9306197 ,  0.93100842,  0.93210893,  0.9338451 ],
       [ 2.13786842,  2.13660992,  2.1362142 ,  2.13657821,  2.13761807],
       [ 3.34469471,  3.34260013,  3.34141999,  3.34104749,  3.34139105],
       [ 4.551521  ,  4.54859035,  4.54662577,  4.54551677,  4.54516402]])

# <font color='red'> Red topic

# Scratch Work

In [ ]:
# Generate a random non-trivial linear program.
m = 15
n = 10
np.random.seed(1)
s0 = np.random.randn(m)
lamb0 = np.maximum(-s0, 0)
s0 = np.maximum(s0, 0)
x0 = np.random.randn(n)
A = np.random.randn(m, n)
b = A @ x0 + s0
c = -A.T @ lamb0

# Define and solve the CVXPY problem.
x = cp.Variable(n)
obj = cp.Minimize(c.T@x)
constr = [A @ x <= b]
prob = cp.Problem(obj, constr)
prob.solve()

# Print result.
print("\nThe optimal value is", prob.value)
print("A solution x is")
print(x.value)
print("A dual solution is")
print(prob.constraints[0].dual_value)

